## Associated legendre polynomials

The associated legendre polynomials are

$$ P_l^m(x) = \frac{(-1)^m}{2^l l!} (1 - x^2)^{m/2} \frac{d^{l+m}}{dx^{l + m}} (x^2 - 1)^l$$

for $m, l \in \mathbf{N}$. For negative orders of $m$, we have

$$ P_l^{-m}(x) = (-1)^m \frac{(l - m)!}{(l + m)!} P_l^m(x) $$

The closed form of $P_l^m(x)$ can be derived as follows:

$$
(x^2 - 1)^l = \sum_{k=0}^l \binom{l}{k} x^{2k} (-1)^{l-k}
$$

$$
\frac{d^{l+m}}{dx^{l+m}} x^{2k} = \frac{(2k)!}{(2k - l - m)!}x^{2k - l - m}
$$

if $l + m \leq 2k$ and $0$ otherwise. Thus, we get:

$$
\frac{d^{l+m}}{dx^{l+m}} (x^2 - 1)^l = \sum_{k=\lceil (l + m)/2 \rceil}^{l} \binom{l}{k} \frac{(2k)!}{(2k - l - m)!}x^{2k - l - m} (-1)^{l-k}
$$

Reindexing $j = l - k$ yields

$$
\frac{d^{l+m}}{dx^{l+m}} (x^2 - 1)^l = \sum_{j=0}^{\lfloor (l-m)/2 \rfloor} \binom{l}{j} \frac{(2l - 2j)!}{(l - 2j - m)!}x^{l - 2j - m} (-1)^j
$$

Thus the closed form to the associated legendre polynomial is 

$$
\begin{align*}

P_l^m(x) &= \frac{(-1)^m}{2^l l!} (1 - x^2)^{m/2} \sum_{j=0}^{\lfloor (l-m)/2 \rfloor} \binom{l}{j} \frac{(2l - 2j)!}{(l - 2j - m)!}x^{l - 2j - m} (-1)^j \\
&= \frac{(-1)^m}{2^l} (1 - x^2)^{m/2} \sum_{j=0}^{\lfloor (l-m)/2 \rfloor} \frac{(-1)^j}{j! (l - j)!} \frac{(2l - 2j)!}{(l - 2j - m)!}x^{l - 2j - m} \\

\end{align*}
$$

## Spherical Harmonics

The SH are defined as 

$$
Y_l^m(\theta, \phi) = (-1)^m \sqrt{\frac{(2l + 1)}{4\pi}\frac{(l-m)!}{(l+m)!}} P_l^m(cos(\theta))e^{im\phi}
$$

Since the SH are complex valued, the real form is defined as

$$
\begin{equation}
Y_l^m = \begin{cases}
        \sqrt{2}(-1)^m \text{Im}(Y_l^{|m|}) & \text{if $m < 0$} \\
        Y_l^0 & \text{if $m = 0$} \\
        \sqrt{2}(-1)^m \text{Re}(Y_l^m) & \text{if $m > 0$}
        \end{cases}
\end{equation}
$$


In [11]:
import sympy as sp
import re

In [12]:
# Matches decimal/scientific notation numbers but not integers
float_re = re.compile(
    r'(?<![\w.])'
    r'([+-]?(?:\d+\.\d*|\.\d+)(?:[eE][+-]?\d+)?'
    r'|[+-]?\d+[eE][+-]?\d+)'
)

def wrap_constants(code):
    return float_re.sub(r'constant<T>(\1)', code)

def to_ccode_vector(expr):
    ev = expr.evalf()
    out = "{\n"

    for i in range(len(ev)):
        code = sp.ccode(sp.simplify(ev[i]))
        code = wrap_constants(code)
        out += f"    {code}"
        if i != len(ev) - 1:
            out += ",\n"

    out += "\n};"
    return out

def to_ccode_matrix(expr):
    ev = sp.Matrix(expr).evalf()

    out = "{\n"

    for i in range(ev.rows):
        row = "    {"
        for j in range(ev.cols):
            code = sp.ccode(sp.simplify(ev[i, j]))
            code = wrap_constants(code)
            row += "" + code
            if j != ev.cols - 1:
                row += ","
            else:
                row += "}"
        out += row
        if i != ev.rows - 1:
            out += ",\n"
    out += "\n};"
    return out

def to_ccode_sym_matrix(expr):
    ev = sp.Matrix(expr).evalf()

    out = "{\n"

    for i in range(ev.rows):
        for j in range(i, ev.cols):
            code = sp.ccode(sp.simplify(ev[i, j]))
            code = wrap_constants(code)
            out += "    " + code
            if i != ev.rows - 1:
                out += ",\n"

    out += "};"
    return out

In [13]:
def legendre(l: int, m: int, x: sp.Expr | float) -> sp.Expr:
    """Returns the symbolic Associated Legendre Polynomial P_l^m(x)."""
    # Create a local dummy variable to construct the polynomial
    dummy_x = sp.Symbol('x')
    expr = sp.assoc_legendre(l, m, dummy_x)
    return expr.subs(dummy_x, x)

def sph_harm(l: int, m: int, theta: sp.Expr | float, phi: sp.Expr | float) -> sp.Expr:
    """Returns the symbolic Spherical Harmonic Y_l^m(theta, phi)."""
    t, p = sp.symbols('theta phi')
    expr = sp.Ynm(l, m, t, p)
    return expr.subs({t: theta, p: phi})

def sph_harm_real(l: int, m: int, theta: sp.Expr | float, phi: sp.Expr | float) -> sp.Expr:
    """Returns the symbolic Real Spherical Harmonic Y_{l,m}(theta, phi)."""
    t, p = sp.symbols('theta phi')
    
    if m < 0:
        # Uses standard linear combination for real spherical harmonics
        expr = sp.sqrt(2) * ((-1) ** m) * sp.im(sp.Ynm(l, abs(m), t, p))
    elif m == 0:
        expr = sp.re(sp.Ynm(l, 0, t, p))
    else:
        expr = sp.sqrt(2) * ((-1) ** m) * sp.re(sp.Ynm(l, m, t, p))
        
    return expr.subs({t: theta, p: phi})

In [19]:
theta = sp.Symbol("theta")
phi = sp.Symbol("phi")

basis = []

for l in [0, 2, 4]:
    for m in range(-l, l + 1):
        basis.append(sph_harm_real(l, m, theta, phi))

basis = sp.Matrix(basis)

# print(to_ccode_vector(basis))
basis

Matrix([
[                                                                                  1/(2*sqrt(pi))],
[                           0.0833333333333333*sqrt(15)*(3 - 3*cos(theta)**2)*sin(2*phi)/sqrt(pi)],
[                               0.5*sqrt(15)*sqrt(1 - cos(theta)**2)*sin(phi)*cos(theta)/sqrt(pi)],
[                                                  sqrt(5)*(3*cos(theta)**2/2 - 1/2)/(2*sqrt(pi))],
[                               sqrt(15)*sqrt(1 - cos(theta)**2)*cos(phi)*cos(theta)/(2*sqrt(pi))],
[                                         sqrt(15)*(3 - 3*cos(theta)**2)*cos(2*phi)/(12*sqrt(pi))],
[                                      0.1875*sqrt(35)*(1 - cos(theta)**2)**2*sin(4*phi)/sqrt(pi)],
[                        0.375*sqrt(70)*(1 - cos(theta)**2)**(3/2)*sin(3*phi)*cos(theta)/sqrt(pi)],
[               0.05*sqrt(5)*(1 - cos(theta)**2)*(105*cos(theta)**2/2 - 15/2)*sin(2*phi)/sqrt(pi)],
[  0.15*sqrt(10)*sqrt(1 - cos(theta)**2)*(35*cos(theta)**3/2 - 15*cos(theta)/2)*sin(phi)/sq